# Attention Is All You Need 재구현 실습 - 추가 실습 코드: Scaled Dot-Product Attention 구현

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section ID: `practice-attention-basic`
- Section: 추가 실습 코드: Scaled Dot-Product Attention 구현


# Integrated Notebook: 02_Scaled_Dot_Product_Attention_Practice

## Original Version
Source: 064_mod-attention-paper-lab_practice-attention-basic_Attention_Is_All_You_Need_#Uc7ac#Uad6c#Ud604_#Uc2e4#Uc2b5_-_#Ucd94#Uac00_#Uc2e4#Uc2b5_#Ucf54#Ub4dc-_Scaled_Dot-Product_Attention_#Uad6c#Ud604.ipynb

# Attention Is All You Need 재구현 실습 - 추가 실습 코드: Scaled Dot-Product Attention 구현

- Tutorial ID: `mod-attention-paper-lab`
- Tutorial: Attention Is All You Need 재구현 실습
- Section: 추가 실습 코드: Scaled Dot-Product Attention 구현

In [ ]:
# ============================================================
# 코드 읽는 법 — 추가 실습 코드: Scaled Dot-Product Attention 구현
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    """수치 안정적인 Softmax"""
    x_max = np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(x - x_max)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    """
    Scaled Dot-Product Attention 구현
    
    Args:
        Q: Query 행렬 (seq_len, d_k)
        K: Key 행렬 (seq_len, d_k)
        V: Value 행렬 (seq_len, d_v)
    
    Returns:
        output: Attention 출력
        weights: Attention 가중치
    """
    d_k = Q.shape[-1]
    
    # Step 1: Q·K^T 계산
        scores = np.matmul(Q, K.T)
    print(f"1. Scores (QK^T) shape: {scores.shape}")
    
    # Step 2: sqrt(d_k)로 스케일링
        scaled_scores = scores / np.sqrt(d_k)
    print(f"2. Scaled scores: 각 값을 √{d_k} = {np.sqrt(d_k):.2f}로 나눔")
    
    # Step 3: Softmax로 확률 분포 변환
        weights = softmax(scaled_scores)
    print(f"3. Attention weights (각 행의 합 = 1):")
    print(weights.round(3))
    
    # Step 4: Value와 가중합
        output = np.matmul(weights, V)
    print(f"4. Output shape: {output.shape}")
    
    return output, weights

# 테스트: 4개 토큰, d_k=d_v=8
np.random.seed(42)
seq_len, d_model = 4, 8

Q = np.random.randn(seq_len, d_model)
K = np.random.randn(seq_len, d_model)
V = np.random.randn(seq_len, d_model)

print("=" * 50)
print("Scaled Dot-Product Attention 실행")
print("=" * 50)
output, attn_weights = scaled_dot_product_attention(Q, K, V)
print("\\n✅ Attention 계산 완료!")

## AI Developed Version 1
Source: 064_Scaled_Dot-Product_Attention_#Ucd94#Uac00#Uc2e4#Uc2b5.ipynb

# 📘 Scaled Dot-Product Attention 구현 - 추가 실습

## "Attention Is All You Need" 재구현 실습 - 심화편

---

### 🎯 학습 목표

이전 노트북(063)에서 Attention의 기본 개념을 배웠습니다.
이번에는 더 깊이 들어가서:

1. **nn.Linear**를 사용한 실전적 구현 (수동 가중치 → PyTorch 레이어)
2. **스케일링의 효과**를 실험으로 확인
3. **Padding Mask**를 적용한 Attention
4. **PyTorch 내장 Attention과 비교**

### 📋 선수 지식

- 노트북 063의 내용 (Q, K, V 개념, Scaled Dot-Product Attention 수식)
- PyTorch 기본 사용법 (텐서 연산, nn.Module)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print(f"PyTorch 버전: {torch.__version__}")

## 실습 1: nn.Linear를 사용한 구현

### nn.Linear란?

이전 노트북에서는 `W_Q = torch.randn(...)` 처럼 수동으로 가중치를 만들었습니다.
실제 PyTorch에서는 **nn.Linear** 레이어를 사용합니다.

```python
# 수동 방식 (이전 노트북)
W_Q = torch.randn(d_model, d_k)
Q = x @ W_Q

# nn.Linear 방식 (실전)
linear_q = nn.Linear(d_model, d_k)
Q = linear_q(x)  # 내부적으로 x @ W + b 수행
```

nn.Linear의 장점:
- 가중치(weight)와 편향(bias)을 자동 관리
- gradient 계산 자동 지원
- optimizer로 학습 가능

In [ ]:
# ============================================================
# nn.Linear를 사용한 Attention 클래스

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """
    Scaled Dot-Product Attention을 nn.Module로 구현합니다.
    
    이 클래스는 다음을 수행합니다:
    1. 입력 x로부터 Q, K, V를 생성 (nn.Linear 사용)
    2. Attention score 계산 (QK^T)
    3. 스케일링 (÷ √d_k)
    4. (선택) Mask 적용
    5. Softmax로 확률 변환
    6. Value에 가중치 적용
    """
    
    def __init__(self, d_model, d_k, d_v):
        """
        Args:
            d_model: 입력 임베딩 차원
            d_k: Query/Key 차원
            d_v: Value 차원
        """
        super().__init__()
        
        # Q, K, V를 만들기 위한 선형 변환 레이어
        # bias=False로 설정하면 순수 행렬 곱만 수행 (편향 없음)
        self.W_Q = nn.Linear(d_model, d_k, bias=False)
        self.W_K = nn.Linear(d_model, d_k, bias=False)
        self.W_V = nn.Linear(d_model, d_v, bias=False)
        
        # 스케일링 팩터 미리 계산
        self.scale = math.sqrt(d_k)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: 입력 텐서, shape = (batch_size, seq_len, d_model)
            mask: 마스크 텐서 (선택), shape = (batch_size, 1, seq_len)
                  0인 위치는 attention에서 무시됩니다.
        
        Returns:
            output: (batch_size, seq_len, d_v)
            attention_weights: (batch_size, seq_len, seq_len)
        """
        # Step 1: Q, K, V 생성
        Q = self.W_Q(x)  # (batch_size, seq_len, d_k)
        K = self.W_K(x)  # (batch_size, seq_len, d_k)
        V = self.W_V(x)  # (batch_size, seq_len, d_v)
        
        # Step 2: Attention Score 계산 + 스케일링
        # K.transpose(-2, -1): 마지막 두 차원을 전치
        # (batch, seq, d_k) → (batch, d_k, seq)
        scores = (Q @ K.transpose(-2, -1)) / self.scale
        # scores shape: (batch_size, seq_len, seq_len)
        
        # Step 3: Mask 적용 (선택적)
        if mask is not None:
            # mask가 0인 위치에 -inf를 넣어서
            # softmax 후 해당 위치의 확률을 0으로 만듭니다
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Step 4: Softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # Step 5: Value에 가중치 적용
        output = attention_weights @ V
        
        return output, attention_weights


# 테스트
d_model, d_k, d_v = 8, 8, 8
batch_size, seq_len = 2, 4

attention = ScaledDotProductAttention(d_model, d_k, d_v)
x = torch.randn(batch_size, seq_len, d_model)

output, weights = attention(x)
print(f"입력 shape:              {x.shape}")
print(f"출력 shape:              {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\n첫 번째 문장의 Attention weights:")
print(weights[0].detach())

## 실습 2: 스케일링의 효과 실험

d_k 값이 커질수록 스케일링이 왜 중요한지 직접 실험해봅시다.

스케일링 없이 d_k가 커지면 어떤 일이 일어나는지 확인합니다.

In [ ]:
# ============================================================
# 실험: 스케일링 유무에 따른 softmax 출력 비교

In [ ]:
print("=" * 60)
print("d_k 크기에 따른 스케일링 효과 비교")
print("=" * 60)

for d_k_test in [4, 16, 64, 256, 1024]:
    # 랜덤 Q, K 생성
    Q_test = torch.randn(1, 4, d_k_test)
    K_test = torch.randn(1, 4, d_k_test)
    
    # 스케일링 없이
    scores_no_scale = Q_test @ K_test.transpose(-2, -1)
    weights_no_scale = F.softmax(scores_no_scale, dim=-1)
    
    # 스케일링 있을 때
    scores_scaled = scores_no_scale / math.sqrt(d_k_test)
    weights_scaled = F.softmax(scores_scaled, dim=-1)
    
    # 분포의 "날카로움" 측정 (최대값)
    # 최대값이 1에 가까우면 = 한 곳에만 집중 (너무 날카로움)
    # 최대값이 0.25에 가까우면 = 균등 분포 (4개 토큰이므로)
    print(f"\nd_k = {d_k_test:4d}:")
    print(f"  스케일링 없음 - 최대 weight: {weights_no_scale.max():.6f}")
    print(f"  스케일링 있음 - 최대 weight: {weights_scaled.max():.6f}")
    print(f"  스케일링 없음 - 분포: {weights_no_scale[0][0].tolist()}")

print("\n" + "=" * 60)
print("[결론]")
print("d_k가 커질수록 스케일링 없는 경우 softmax가 one-hot에 가까워집니다.")
print("이러면 gradient가 거의 0이 되어 학습이 어려워집니다!")
print("√d_k로 나눠주면 적절한 분포를 유지할 수 있습니다.")

## 실습 3: Padding Mask 적용

### Padding이란?

실제 데이터에서는 문장 길이가 다릅니다:
- 문장 1: "나는 고양이를 좋아한다" (4 토큰)
- 문장 2: "안녕" (1 토큰)

배치 처리를 위해 짧은 문장에 [PAD] 토큰을 추가합니다:
- 문장 1: ["나는", "고양이를", "좋아", "한다"]
- 문장 2: ["안녕", "[PAD]", "[PAD]", "[PAD]"]

**[PAD] 토큰은 의미가 없으므로** Attention에서 무시해야 합니다!
이를 위해 **Padding Mask**를 사용합니다.

In [ ]:
# ============================================================
# Padding Mask 예시

In [ ]:
# 배치 크기 2, 시퀀스 길이 4
# 문장 1: 모든 토큰 유효 (mask = [1, 1, 1, 1])
# 문장 2: 첫 번째만 유효, 나머지는 PAD (mask = [1, 0, 0, 0])

# Padding mask 생성
# 1 = 유효한 토큰, 0 = 패딩 토큰
padding_mask = torch.tensor([
    [1, 1, 1, 1],  # 문장 1: 4개 토큰 모두 유효
    [1, 0, 0, 0],  # 문장 2: 1개만 유효, 3개는 PAD
])

print("Padding Mask:")
print(padding_mask)
print(f"Shape: {padding_mask.shape}")

# Attention에서 사용할 형태로 변환
# (batch, seq) → (batch, 1, seq) : broadcasting을 위해
# 각 Query 토큰이 모든 Key 토큰에 대해 같은 mask를 적용
padding_mask_expanded = padding_mask.unsqueeze(1)
print(f"\n확장된 mask shape: {padding_mask_expanded.shape}")
print("확장된 mask:")
print(padding_mask_expanded)

In [ ]:
# ============================================================
# Padding Mask를 적용한 Attention

In [ ]:
# 입력 생성
x_padded = torch.randn(2, 4, d_model)

# Mask 없이 Attention 수행
output_no_mask, weights_no_mask = attention(x_padded, mask=None)

# Mask 적용하여 Attention 수행
output_masked, weights_masked = attention(x_padded, mask=padding_mask_expanded)

# 결과 비교 (문장 2에 주목)
print("문장 2의 Attention Weights 비교")
print("\n[Mask 없이] - PAD 토큰에도 attention이 분산됨 (잘못됨!):")
print(weights_no_mask[1].detach())

print("\n[Mask 적용] - PAD 토큰의 attention이 0 (올바름!):")
print(weights_masked[1].detach())

print("\n[해석]")
print("Mask 적용 후, PAD 위치(index 1,2,3)의 가중치가 0이 됩니다.")
print("모든 attention이 유효한 토큰(index 0)에만 집중됩니다.")

## 실습 4: Attention 시각화 비교

Mask 유무에 따른 Attention weights를 나란히 시각화합니다.

In [ ]:
# ============================================================
# Mask 유무에 따른 Attention 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

tokens_2 = ["안녕", "[PAD]", "[PAD]", "[PAD]"]

for idx, (title, w) in enumerate([
    ("Mask 없음 (문장 2)", weights_no_mask[1].detach().numpy()),
    ("Mask 적용 (문장 2)", weights_masked[1].detach().numpy())
]):
    ax = axes[idx]
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.set_xticklabels(tokens_2)
    ax.set_yticklabels(tokens_2)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Key")
    ax.set_ylabel("Query")
    
    for i in range(4):
        for j in range(4):
            color = "white" if w[i, j] > 0.5 else "black"
            ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center",
                    fontsize=10, color=color)
    
    plt.colorbar(im, ax=ax)

plt.suptitle("Padding Mask의 효과", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 실습 5: PyTorch 내장 Attention과 비교

PyTorch는 `F.scaled_dot_product_attention`이라는 내장 함수를 제공합니다.
우리가 구현한 것과 동일한 결과가 나오는지 확인해봅시다.

In [ ]:
# ============================================================
# 우리 구현 vs PyTorch 내장 함수 비교

In [ ]:
# 동일한 Q, K, V 사용
torch.manual_seed(123)
Q_compare = torch.randn(1, 4, 8)
K_compare = torch.randn(1, 4, 8)
V_compare = torch.randn(1, 4, 8)

# --- 우리의 구현 ---
def our_attention(Q, K, V):
    d_k = K.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    weights = F.softmax(scores, dim=-1)
    output = weights @ V
    return output

our_output = our_attention(Q_compare, K_compare, V_compare)

# --- PyTorch 내장 함수 ---
# PyTorch 2.0+ 에서 사용 가능
try:
    pytorch_output = F.scaled_dot_product_attention(Q_compare, K_compare, V_compare)
    
    # 비교
    diff = (our_output - pytorch_output).abs().max()
    print(f"우리 구현 출력 shape: {our_output.shape}")
    print(f"PyTorch 내장 출력 shape: {pytorch_output.shape}")
    print(f"\n최대 차이: {diff:.10f}")
    print(f"동일한 결과: {diff < 1e-6}")
    print("\n✅ 우리의 구현이 PyTorch 내장 함수와 동일한 결과를 생성합니다!")
except AttributeError:
    print("PyTorch 2.0+ 이 필요합니다. 하지만 우리 구현은 정확합니다!")
    print(f"우리 구현 출력 shape: {our_output.shape}")
    print(our_output)

## 실습 6: 실제 단어로 Attention 이해하기

간단한 임베딩을 사용하여 실제 단어들의 Attention을 관찰해봅시다.

In [ ]:
# ============================================================
# 간단한 단어 임베딩으로 Attention 관찰

In [ ]:
# 간단한 단어 사전과 임베딩
# 비슷한 의미의 단어끼리 비슷한 벡터를 갖도록 설정
word_embeddings = {
    "고양이":  torch.tensor([1.0, 0.8, 0.1, 0.2, 0.0, 0.0, 0.1, 0.0]),
    "강아지":  torch.tensor([0.9, 0.9, 0.2, 0.1, 0.0, 0.0, 0.2, 0.0]),
    "먹다":    torch.tensor([0.0, 0.1, 0.9, 0.8, 0.1, 0.0, 0.0, 0.0]),
    "사료":    torch.tensor([0.1, 0.2, 0.8, 0.7, 0.3, 0.0, 0.0, 0.0]),
    "자동차":  torch.tensor([0.0, 0.0, 0.0, 0.1, 0.9, 0.8, 0.0, 0.1]),
}

# 문장: "고양이 강아지 먹다 사료"
sentence_words = ["고양이", "강아지", "먹다", "사료"]
sentence_emb = torch.stack([word_embeddings[w] for w in sentence_words]).unsqueeze(0)

print(f"문장: {' '.join(sentence_words)}")
print(f"임베딩 shape: {sentence_emb.shape}")

# 단순 Self-Attention (가중치 변환 없이, Q=K=V=입력)
# 이렇게 하면 순수하게 단어 벡터 간 유사도만 봅니다
scores_simple = sentence_emb @ sentence_emb.transpose(-2, -1)
scores_scaled_simple = scores_simple / math.sqrt(8)
weights_simple = F.softmax(scores_scaled_simple, dim=-1)

print("\nAttention Weights:")
for i, w_from in enumerate(sentence_words):
    print(f"  {w_from:6s} → ", end="")
    for j, w_to in enumerate(sentence_words):
        print(f"{w_to}:{weights_simple[0, i, j]:.3f}  ", end="")
    print()

print("\n[해석]")
print("  - '고양이'와 '강아지'는 비슷한 임베딩 → 서로에게 높은 attention")
print("  - '먹다'와 '사료'는 비슷한 임베딩 → 서로에게 높은 attention")
print("  - '고양이'↔'사료'는 다른 영역 → 낮은 attention")

## 🔑 핵심 정리

### 이번 노트북에서 배운 것

1. **nn.Linear 활용**: 실전에서는 `nn.Linear`로 Q, K, V를 만듭니다
2. **스케일링의 중요성**: d_k가 클수록 스케일링이 필수적입니다
3. **Padding Mask**: PAD 토큰을 무시하기 위해 mask를 사용합니다
4. **PyTorch 내장 함수**: 우리 구현이 공식 구현과 동일한 결과를 냅니다

### Mask 종류 정리

| Mask 종류 | 용도 | 적용 위치 |
|----------|------|----------|
| **Padding Mask** | PAD 토큰 무시 | Encoder, Decoder |
| **Causal Mask** | 미래 토큰 차단 | Decoder만 |

다음 노트북(065)에서 Causal Mask를 구현합니다!

## AI Developed Version 2
Source: 02_Scaled_Dot_Product_Attention_Practice.ipynb

# 📚 Notebook 2: Scaled Dot-Product Attention 심화 실습

## "Attention Is All You Need" 재구현 - 추가 실습

---

### 🎯 학습 목표
Notebook 1에서 배운 Scaled Dot-Product Attention을 **더 깊이 이해**하기 위한 실습입니다.

### 📋 실습 내용
1. 실제 단어 임베딩으로 Attention 실험
2. Attention 가중치의 의미 분석
3. Padding Mask 구현 및 실습
4. 다양한 d_k 값에 따른 Attention 변화 관찰
5. Self-Attention vs Cross-Attention 이해

### ⚠️ 사전 지식
- Notebook 1 (Scaled Dot-Product Attention) 완료 필수

---

In [ ]:
# ============================================================
# 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

print("✅ 라이브러리 로딩 완료!")

In [ ]:
# ============================================================
# Notebook 1에서 만든 함수 (복습)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    수식: Attention(Q,K,V) = softmax(QK^T / √d_k) V
    """
    d_k = K.size(-1)
    
    # Step 1: 유사도 점수 계산
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: 스케일링
    scores = scores / math.sqrt(d_k)
    
    # Step 2.5: 마스크 적용 (선택)
    if mask is not None:
        # mask가 0인 위치를 -inf로 채워서 softmax 후 0이 되게 함
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 3: Softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Value에 가중치 적용
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

print("✅ scaled_dot_product_attention 함수 준비 완료")

## 실습 1: 실제 단어 임베딩으로 Attention 실험 🧪

### 간단한 단어 임베딩 테이블 만들기

실제 모델에서는 각 단어가 고유한 벡터(임베딩)로 표현됩니다.
여기서는 간단한 단어 사전과 임베딩을 직접 만들어 보겠습니다.

**핵심 개념: 임베딩(Embedding)이란?**
- 단어를 고정 길이의 숫자 벡터로 변환하는 것
- 비슷한 의미의 단어는 비슷한 벡터를 가짐
- 예: "고양이"와 "강아지"는 비슷한 벡터, "고양이"와 "자동차"는 다른 벡터

In [ ]:
# ============================================================
# 간단한 단어 임베딩 생성

In [ ]:
# 단어 사전 정의 (간단한 예시)
vocab = {
    '고양이': 0, '강아지': 1, '물고기': 2,
    '좋아한다': 3, '싫어한다': 4, '먹는다': 5,
    '나는': 6, '너는': 7, '를': 8
}

d_model = 16  # 임베딩 차원

# PyTorch의 Embedding 레이어 사용
# nn.Embedding(단어 수, 임베딩 차원)
embedding = nn.Embedding(len(vocab), d_model)

# 문장 1: "나는 고양이를 좋아한다"
sentence1_ids = torch.tensor([vocab['나는'], vocab['고양이'], vocab['를'], vocab['좋아한다']])
sentence1_emb = embedding(sentence1_ids)  # (4, 16)

print("문장: 나는 고양이를 좋아한다")
print(f"토큰 ID: {sentence1_ids.tolist()}")
print(f"임베딩 shape: {sentence1_emb.shape}")
print(f"\n각 단어의 임베딩 벡터 크기:")
for word, idx in [('나는', 0), ('고양이', 1), ('를', 2), ('좋아한다', 3)]:
    print(f"  {word}: [{sentence1_emb[idx][:4].tolist()}, ...]  (앞 4개만 표시)")

In [ ]:
# ============================================================
# Self-Attention: 같은 문장 내에서 서로 주목하기

In [ ]:
# Self-Attention에서는 Q, K, V가 모두 같은 입력에서 옵니다
# 실제로는 각각 다른 Linear 레이어를 통과하지만,
# 여기서는 간단히 입력 자체를 Q, K, V로 사용합니다

# Q, K, V를 위한 Linear 변환
d_k = 8
d_v = 8
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_v, bias=False)

# 임베딩을 Q, K, V로 변환
Q = W_Q(sentence1_emb)  # (4, 16) → (4, 8)
K = W_K(sentence1_emb)  # (4, 16) → (4, 8)
V = W_V(sentence1_emb)  # (4, 16) → (4, 8)

print("Self-Attention: Q, K, V 모두 같은 문장에서 생성")
print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

# Attention 계산
output, weights = scaled_dot_product_attention(Q, K, V)

print(f"\nAttention 출력 shape: {output.shape}")
print(f"\nAttention 가중치:")
tokens = ['나는', '고양이', '를', '좋아한다']
for i, token in enumerate(tokens):
    w = weights[i].detach().numpy()
    pairs = [f"{tokens[j]}({w[j]:.3f})" for j in range(len(tokens))]
    print(f"  {token} → {', '.join(pairs)}")

## 실습 2: Padding Mask 이해와 구현 🎭

### Padding이란?

배치 처리 시 모든 문장의 길이를 맞춰야 합니다.
짧은 문장은 **PAD 토큰**으로 채웁니다.

```
문장 1: "나는 고양이를 좋아한다"     → [나는, 고양이, 를, 좋아한다]
문장 2: "강아지가 좋다"              → [강아지, 좋다, PAD, PAD]
```

**문제:** PAD 토큰은 의미가 없는데, Attention이 PAD에도 주목하면 안 됩니다!

**해결:** Padding Mask로 PAD 위치의 Attention 가중치를 0으로 만듭니다.

In [ ]:
# ============================================================
# Padding Mask 구현

In [ ]:
# 두 문장을 배치로 처리
# 문장 1: 길이 4 → "나는 고양이 를 좋아한다"
# 문장 2: 길이 2 → "강아지 좋아한다" + PAD PAD

# 배치 입력 시뮬레이션
batch_size = 2
max_seq_len = 4

# 임의의 Q, K, V 생성 (배치 포함)
Q_batch = torch.randn(batch_size, max_seq_len, d_k)
K_batch = torch.randn(batch_size, max_seq_len, d_k)
V_batch = torch.randn(batch_size, max_seq_len, d_v)

# Padding Mask 생성
# 1 = 실제 토큰 (주목해도 됨)
# 0 = PAD 토큰 (주목하면 안 됨)
padding_mask = torch.tensor([
    [1, 1, 1, 1],  # 문장 1: 4개 모두 실제 토큰
    [1, 1, 0, 0],  # 문장 2: 2개만 실제, 나머지 PAD
])

# Attention에서 사용하려면 (batch, 1, seq_len) 또는 (batch, seq_len, seq_len) 형태가 필요
# unsqueeze(1)로 차원 추가: (2, 4) → (2, 1, 4)
# 이렇게 하면 모든 Query 위치에서 동일한 마스크가 적용됨
attn_mask = padding_mask.unsqueeze(1)  # (2, 1, 4)

print("Padding Mask:")
print(padding_mask)
print(f"\nAttention용 Mask shape: {attn_mask.shape}")
print(attn_mask)

In [ ]:
# ============================================================
# Mask 적용 전후 비교

In [ ]:
# 마스크 없이 Attention
output_no_mask, weights_no_mask = scaled_dot_product_attention(Q_batch, K_batch, V_batch)

# 마스크 있는 Attention  
output_masked, weights_masked = scaled_dot_product_attention(Q_batch, K_batch, V_batch, mask=attn_mask)

print("📊 문장 2의 Attention 가중치 비교 (PAD가 있는 문장)")
print("=" * 60)

labels = ['강아지', '좋아한다', 'PAD', 'PAD']

print("\n❌ 마스크 없이 (PAD에도 주목함 - 잘못된 경우):")
for i, label in enumerate(labels):
    w = weights_no_mask[1, i].detach().numpy()  # 문장 2 (index 1)
    parts = [f"{labels[j]}({w[j]:.3f})" for j in range(max_seq_len)]
    print(f"  {label} → {', '.join(parts)}")

print("\n✅ 마스크 적용 (PAD에 주목하지 않음 - 올바른 경우):")
for i, label in enumerate(labels):
    w = weights_masked[1, i].detach().numpy()  # 문장 2 (index 1)
    parts = [f"{labels[j]}({w[j]:.3f})" for j in range(max_seq_len)]
    print(f"  {label} → {', '.join(parts)}")

print("\n💡 마스크 적용 후: PAD 위치의 가중치가 0.000이 된 것을 확인!")

## 실습 3: Self-Attention vs Cross-Attention 비교 🔄

### Self-Attention
- Q, K, V가 **모두 같은 시퀀스**에서 나옴
- 문장 내 단어들 사이의 관계를 파악
- Encoder와 Decoder 모두에서 사용

### Cross-Attention
- Q는 한 시퀀스에서, K와 V는 **다른 시퀀스**에서 나옴
- 번역에서: Q=디코더(출력 문장), K&V=인코더(입력 문장)
- Decoder에서 Encoder의 정보를 가져올 때 사용

In [ ]:
# ============================================================
# Self-Attention vs Cross-Attention 비교

In [ ]:
# 시나리오: 영어 → 한국어 번역
# 영어 (Encoder 입력): "I love cats" (3 tokens)
# 한국어 (Decoder 출력): "나는 고양이를 좋아한다" (4 tokens)

encoder_seq_len = 3  # 영어 문장 길이
decoder_seq_len = 4  # 한국어 문장 길이

# 가상의 Encoder 출력과 Decoder 입력
encoder_output = torch.randn(encoder_seq_len, d_k)  # 영어 쪽 표현
decoder_input = torch.randn(decoder_seq_len, d_k)   # 한국어 쪽 표현

print("🔵 Self-Attention (Encoder 내부)")
print("  Q, K, V 모두 Encoder에서 옴")
Q_self = encoder_output  # Q = Encoder
K_self = encoder_output  # K = Encoder
V_self = encoder_output  # V = Encoder
out_self, w_self = scaled_dot_product_attention(Q_self, K_self, V_self)
print(f"  가중치 shape: {w_self.shape}  → ({encoder_seq_len}x{encoder_seq_len})")
print(f"  = 각 영어 단어가 다른 영어 단어들에 얼마나 주목하는지\n")

print("🟢 Cross-Attention (Decoder → Encoder)")
print("  Q = Decoder에서, K와 V = Encoder에서 옴")
Q_cross = decoder_input   # Q = Decoder (한국어)
K_cross = encoder_output  # K = Encoder (영어)
V_cross = encoder_output  # V = Encoder (영어)
out_cross, w_cross = scaled_dot_product_attention(Q_cross, K_cross, V_cross)
print(f"  가중치 shape: {w_cross.shape}  → ({decoder_seq_len}x{encoder_seq_len})")
print(f"  = 각 한국어 단어가 각 영어 단어에 얼마나 주목하는지")

print(f"\n💡 Cross-Attention의 가중치 행렬이 직사각형인 것에 주목!")
print(f"   Self-Attention: {encoder_seq_len}x{encoder_seq_len} (정사각형)")
print(f"   Cross-Attention: {decoder_seq_len}x{encoder_seq_len} (직사각형)")

In [ ]:
# ============================================================
# Cross-Attention 가중치 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Self-Attention 시각화
en_tokens = ['I', 'love', 'cats']
im1 = axes[0].imshow(w_self.detach().numpy(), cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(encoder_seq_len))
axes[0].set_yticks(range(encoder_seq_len))
axes[0].set_xticklabels(en_tokens)
axes[0].set_yticklabels(en_tokens)
axes[0].set_title('Self-Attention (Encoder)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Key')
axes[0].set_ylabel('Query')
for i in range(encoder_seq_len):
    for j in range(encoder_seq_len):
        axes[0].text(j, i, f'{w_self[i,j]:.2f}', ha='center', va='center')

# Cross-Attention 시각화
ko_tokens = ['나는', '고양이를', '좋아', '한다']
im2 = axes[1].imshow(w_cross.detach().numpy(), cmap='Greens', vmin=0, vmax=1)
axes[1].set_xticks(range(encoder_seq_len))
axes[1].set_yticks(range(decoder_seq_len))
axes[1].set_xticklabels(en_tokens)
axes[1].set_yticklabels(ko_tokens)
axes[1].set_title('Cross-Attention (Decoder→Encoder)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Key (English)')
axes[1].set_ylabel('Query (Korean)')
for i in range(decoder_seq_len):
    for j in range(encoder_seq_len):
        axes[1].text(j, i, f'{w_cross[i,j]:.2f}', ha='center', va='center')

plt.tight_layout()
plt.show()

print("💡 Cross-Attention에서 '고양이를'이 'cats'에 높은 가중치를 보이면 좋은 번역 모델!")

## 실습 4: d_k 값에 따른 Attention 변화 🔎

d_k(Key/Query 차원)가 Attention에 어떤 영향을 미치는지 실험합니다.

In [ ]:
# ============================================================
# d_k에 따른 Attention 분포 변화 실험

In [ ]:
seq_len = 6
d_k_values = [2, 8, 32, 128, 512]

fig, axes = plt.subplots(1, len(d_k_values), figsize=(20, 4))

for idx, test_dk in enumerate(d_k_values):
    torch.manual_seed(42)  # 공정한 비교를 위해 시드 고정
    Q_test = torch.randn(seq_len, test_dk)
    K_test = torch.randn(seq_len, test_dk)
    V_test = torch.randn(seq_len, test_dk)
    
    _, w = scaled_dot_product_attention(Q_test, K_test, V_test)
    
    axes[idx].imshow(w.detach().numpy(), cmap='Reds', vmin=0, vmax=1)
    axes[idx].set_title(f'd_k = {test_dk}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Key')
    if idx == 0:
        axes[idx].set_ylabel('Query')

plt.suptitle('d_k 값에 따른 Attention 가중치 분포', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("💡 d_k가 작으면 가중치가 더 균등하게 분포 (soft attention)")
print("   d_k가 크면 특정 위치에 집중 (sharp attention)")
print("   실제 Transformer에서는 d_k = d_model / num_heads 로 설정합니다.")

## 📝 핵심 정리

| 개념 | 설명 |
|------|------|
| **Self-Attention** | Q, K, V 모두 같은 시퀀스 → 문장 내 단어 간 관계 파악 |
| **Cross-Attention** | Q는 한 시퀀스, K/V는 다른 시퀀스 → 두 시퀀스 간 관계 |
| **Padding Mask** | PAD 토큰을 무시하도록 하는 마스크 |
| **d_k의 영향** | 작을수록 균등, 클수록 집중적인 Attention |

### 다음 노트북 예고 🔜
Notebook 3에서는 **Causal (Masked) Attention**을 구현합니다.
→ Decoder에서 미래 토큰을 볼 수 없도록 하는 방법!

## AI Developed Version 3
Source: 064_Practice_Scaled_Dot_Product_Attention.ipynb

# 📝 실습 064: Scaled Dot-Product Attention 직접 구현해보기

## 🎯 목표
앞선 063 실습에서 배운 내용을 바탕으로, **직접 Attention을 구현**해봅니다.

> ⚠️ 이 노트북은 **실습용**입니다. `# TODO:` 부분을 직접 채워보세요!
> 아래에 정답 코드도 포함되어 있으니 막히면 참고하세요.

### 다룰 내용
1. Attention 함수 직접 구현 (빈 칸 채우기)
2. 학습 가능한 W_Q, W_K, W_V 행렬 추가
3. 다양한 문장 길이 처리 (Encoder-Decoder Attention)
4. 패딩(Padding) 마스크 구현
5. Temperature를 이용한 Attention 날카로움 조절

---

### 복습: Scaled Dot-Product Attention 공식
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

torch.manual_seed(42)
print("✅ 라이브러리 로드 완료")

## 🏋️ 실습 1: Attention 함수 직접 구현

아래 `# TODO` 부분을 채워보세요!

In [ ]:
# ============================================================
# 🏋️ 실습 1: Attention 함수 직접 구현해보기

In [ ]:
def my_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention 직접 구현 실습
    
    힌트:
    1. d_k = Q의 마지막 차원 크기 (Q.size(-1))
    2. scores = Q와 K를 내적 후 sqrt(d_k)로 나누기
       - K를 전치할 때: K.transpose(-2, -1) 사용
       - 행렬 곱: torch.matmul(A, B)
    3. mask가 있으면 scores에서 mask=1인 위치를 -1e9로 설정
       - masked_fill(condition, value) 사용
    4. softmax를 마지막 차원에 적용
       - F.softmax(x, dim=-1)
    5. attention_weights와 V를 행렬 곱
    """
    
    # TODO 1: d_k 가져오기
    # d_k = ???
    d_k = Q.size(-1)  # 정답
    
    # TODO 2: scores 계산 (QK^T / sqrt(d_k))
    # scores = ???
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)  # 정답
    
    # TODO 3: 마스크 적용 (mask가 None이 아닐 때만)
    # if mask is not None:
    #     scores = ???
    if mask is not None:  # 정답
        scores = scores.masked_fill(mask == 1, -1e9)
    
    # TODO 4: softmax 적용해서 attention_weights 계산
    # attention_weights = ???
    attention_weights = F.softmax(scores, dim=-1)  # 정답
    
    # TODO 5: Value 가중 평균 계산
    # output = ???
    output = torch.matmul(attention_weights, V)  # 정답
    
    return output, attention_weights


# ✅ 구현 검증
print("=" * 60)
print("구현 검증")
print("=" * 60)

batch, seq, d = 2, 4, 8
Q_t = torch.randn(batch, seq, d)
K_t = torch.randn(batch, seq, d)
V_t = torch.randn(batch, seq, d)

out, wts = my_attention(Q_t, K_t, V_t)

print(f"출력 shape: {out.shape}  (예상: torch.Size([{batch}, {seq}, {d}]))")
print(f"가중치 shape: {wts.shape}  (예상: torch.Size([{batch}, {seq}, {seq}]))")
print(f"가중치 합계 (모두 1.0이어야 함): {wts.sum(dim=-1).mean().item():.4f}")
print("\n✅ 정상 동작!" if abs(wts.sum(dim=-1).mean().item() - 1.0) < 1e-5 else "❌ 오류 있음!")

In [ ]:
# ============================================================
# 🏋️ 실습 2: 학습 가능한 W_Q, W_K, W_V 추가

In [ ]:
#
# 앞서 만든 함수는 이미 Q, K, V가 주어진다고 가정했습니다.
# 실제 Transformer는 입력 X를 받아서
# W_Q, W_K, W_V 가중치 행렬로 Q, K, V를 생성합니다.
#
# X: 입력 임베딩  (batch, seq_len, d_model)
# Q = X @ W_Q    (batch, seq_len, d_k)
# K = X @ W_K    (batch, seq_len, d_k)
# V = X @ W_V    (batch, seq_len, d_v)

class SingleHeadAttention(nn.Module):
    """
    학습 가능한 W_Q, W_K, W_V가 포함된 Single-Head Attention
    (Multi-Head Attention의 헤드 1개 버전)
    """
    
    def __init__(self, d_model, d_k, d_v):
        """
        Args:
            d_model: 입력 임베딩 차원 (예: 512)
            d_k:     Query/Key 벡터 차원 (예: 64)
            d_v:     Value 벡터 차원 (예: 64)
        """
        super(SingleHeadAttention, self).__init__()
        
        # 학습 가능한 선형 변환 레이어
        # nn.Linear(in_features, out_features, bias=False)
        # → X @ W^T (내부적으로 전치 적용)
        self.W_Q = nn.Linear(d_model, d_k, bias=False)  # 쿼리 변환
        self.W_K = nn.Linear(d_model, d_k, bias=False)  # 키 변환
        self.W_V = nn.Linear(d_model, d_v, bias=False)  # 값 변환
        
        self.d_k = d_k
        
        # 파라미터 초기화 (Xavier 초기화로 학습 안정성 향상)
        # Xavier 초기화: 입출력 차원에 맞게 가중치 범위 조정
        nn.init.xavier_uniform_(self.W_Q.weight)
        nn.init.xavier_uniform_(self.W_K.weight)
        nn.init.xavier_uniform_(self.W_V.weight)
    
    def forward(self, x, mask=None):
        """
        Args:
            x:    입력  shape = (batch, seq_len, d_model)
            mask: 마스크 shape = (batch, seq_len, seq_len) or None
        Returns:
            output:  shape = (batch, seq_len, d_v)
            weights: shape = (batch, seq_len, seq_len)
        """
        # 입력 X를 Q, K, V로 변환
        # shape: (batch, seq_len, d_model) → (batch, seq_len, d_k)
        Q = self.W_Q(x)   # X @ W_Q^T
        K = self.W_K(x)   # X @ W_K^T
        V = self.W_V(x)   # X @ W_V^T
        
        # Attention 계산
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 1, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        output = torch.matmul(weights, V)
        
        return output, weights


# 테스트
d_model, d_k, d_v = 512, 64, 64
batch_size, seq_len = 3, 10

attn = SingleHeadAttention(d_model=d_model, d_k=d_k, d_v=d_v)

# 입력: 랜덤 임베딩 (실제로는 단어 임베딩)
x = torch.randn(batch_size, seq_len, d_model)

output, weights = attn(x)

print("=" * 60)
print("SingleHeadAttention (학습 가능한 W_Q, W_K, W_V 포함)")
print("=" * 60)
print(f"입력 x: {x.shape}")
print(f"출력 output: {output.shape}")
print(f"가중치 weights: {weights.shape}")
print()

# 총 파라미터 수 계산
total_params = sum(p.numel() for p in attn.parameters())
print(f"학습 파라미터 수: {total_params:,}")
print(f"  W_Q: {d_model} × {d_k} = {d_model*d_k:,}")
print(f"  W_K: {d_model} × {d_k} = {d_model*d_k:,}")
print(f"  W_V: {d_model} × {d_v} = {d_model*d_v:,}")
print(f"  합계: {d_model*d_k + d_model*d_k + d_model*d_v:,}")

In [ ]:
# ============================================================
# 🏋️ 실습 3: Cross-Attention (Encoder-Decoder Attention)

In [ ]:
#
# 지금까지는 Self-Attention (Q=K=V=X) 이었습니다.
# 번역 모델에서는 Cross-Attention도 사용합니다:
#   - Decoder의 Q (디코더가 생성 중인 내용)
#   - Encoder의 K, V (인코더가 처리한 입력 문장 정보)
#
# 예: "나는 사과를 먹었다" → "I ate an apple"
# "ate"를 생성할 때, 인코더의 "먹었다"에 집중!

print("=" * 60)
print("Cross-Attention 예제")
print("(다른 길이의 Q와 K, V 처리)")
print("=" * 60)

# 인코더 출력: 한국어 문장 "나는 사과를 먹었다" (3 토큰)
encoder_seq_len = 3   # "나는", "사과를", "먹었다"

# 디코더: 현재 영어 토큰 생성 중 ("I", "ate" = 2 토큰)
decoder_seq_len = 2   # "I", "ate"

d_model = 64
batch_size = 2

# 인코더 출력 (K, V의 소스)
encoder_output = torch.randn(batch_size, encoder_seq_len, d_model)

# 디코더 현재 상태 (Q의 소스)
decoder_state = torch.randn(batch_size, decoder_seq_len, d_model)

print(f"인코더 출력 (K, V 소스): {encoder_output.shape}")
print(f"  → 한국어 3개 토큰")
print(f"디코더 상태 (Q 소스): {decoder_state.shape}")
print(f"  → 영어 2개 토큰\n")

# W_Q, W_K, W_V 레이어
d_k = 32
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

# Cross-Attention:
# Q는 디코더에서, K와 V는 인코더에서!
Q = W_Q(decoder_state)    # (batch, decoder_seq=2, d_k=32)
K = W_K(encoder_output)   # (batch, encoder_seq=3, d_k=32)
V = W_V(encoder_output)   # (batch, encoder_seq=3, d_k=32)

print(f"Q (from decoder): {Q.shape}  ← (batch=2, decoder_seq=2, d_k=32)")
print(f"K (from encoder): {K.shape}  ← (batch=2, encoder_seq=3, d_k=32)")
print(f"V (from encoder): {V.shape}  ← (batch=2, encoder_seq=3, d_k=32)")

# Cross-Attention 계산
# scores shape: (batch, decoder_seq=2, encoder_seq=3)
# 각 디코더 토큰이 각 인코더 토큰에 얼마나 주목하는지!
output, weights = my_attention(Q, K, V)

print(f"\nCross-Attention 출력: {output.shape}")
print(f"  ← (batch=2, decoder_seq=2, d_k=32)")
print(f"Attention 가중치: {weights.shape}")
print(f"  ← (batch=2, decoder_seq=2, encoder_seq=3)")
print()
print("가중치 해석 (첫 번째 배치):")
print(f"  '영어 토큰 1'이 각 한국어에 주목하는 비율: {weights[0, 0].detach().tolist()}")
print(f"  '영어 토큰 2'이 각 한국어에 주목하는 비율: {weights[0, 1].detach().tolist()}")

In [ ]:
# ============================================================
# 🏋️ 실습 4: 패딩(Padding) 마스크 구현

In [ ]:
#
# 실제 배치 처리에서는 문장 길이가 다를 수 있습니다.
# 짧은 문장은 패딩([PAD])으로 채웁니다.
# 패딩 위치는 Attention에서 무시해야 합니다!
#
# 예:
# 문장1: ["나는", "사과", "먹었다", "<PAD>", "<PAD>"] → 실제 길이=3
# 문장2: ["고양이", "는", "귀엽다", "<PAD>", "<PAD>"] → 실제 길이=3
# 문장3: ["AI", "가", "미래", "를", "바꾼다"]        → 실제 길이=5

def create_padding_mask(seq_lengths, max_len):
    """
    패딩 마스크 생성
    
    Args:
        seq_lengths: 각 문장의 실제 길이 리스트 (패딩 제외)
                     예: [3, 3, 5]
        max_len:     최대 시퀀스 길이 (패딩 포함)
                     예: 5
    
    Returns:
        mask: shape = (batch, 1, max_len)
              패딩 위치 = 1, 실제 위치 = 0
              (1이 True = 마스킹됨 = 무시)
    """
    batch_size = len(seq_lengths)
    
    # 각 위치가 패딩인지 아닌지를 나타내는 마스크
    # shape: (batch_size, max_len)
    # range(max_len): [0, 1, 2, ..., max_len-1]
    mask = torch.zeros(batch_size, max_len, dtype=torch.long)
    
    for i, length in enumerate(seq_lengths):
        # length 이후는 패딩 → 마스크 = 1 (무시)
        if length < max_len:
            mask[i, length:] = 1
    
    # (batch, 1, max_len): 나중에 브로드캐스팅을 위해 차원 추가
    return mask.unsqueeze(1)


# 테스트
seq_lengths = [3, 3, 5]   # 세 문장의 실제 길이
max_len = 5               # 최대 길이 (배치에서 가장 긴 문장)

padding_mask = create_padding_mask(seq_lengths, max_len)

print("=" * 60)
print("패딩 마스크 예시")
print("=" * 60)
print(f"\n문장별 실제 길이: {seq_lengths}")
print(f"최대 길이: {max_len}")
print(f"\n패딩 마스크 shape: {padding_mask.shape}")
print(f"  (1=무시할 패딩, 0=실제 토큰)")
print()
for i, length in enumerate(seq_lengths):
    mask_row = padding_mask[i, 0].tolist()
    print(f"  문장{i+1} (실제길이={length}): {mask_row}")
    print(f"           {'실제' * length + 'PAD' * (max_len - length)}")

# 마스크 시각화
fig, ax = plt.subplots(figsize=(6, 3))
sns.heatmap(
    padding_mask.squeeze(1).numpy(),
    annot=True,
    cmap='RdYlGn_r',
    xticklabels=[f'pos{i}' for i in range(max_len)],
    yticklabels=[f'문장{i+1}' for i in range(3)],
    ax=ax,
    vmin=0, vmax=1
)
ax.set_title('패딩 마스크\n(빨강=패딩/무시, 초록=실제 토큰)', fontsize=11)
plt.tight_layout()
plt.savefig('padding_mask.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 🏋️ 실습 5: Temperature로 Attention 날카로움 조절

In [ ]:
#
# Temperature(온도)는 softmax의 '선명도'를 조절합니다.
#
# Attention(Q,K,V) = softmax(QK^T / (√d_k × temperature)) × V
#
# - temperature < 1: 더 날카로운 집중 (한 곳에 집중)
# - temperature = 1: 표준 Attention
# - temperature > 1: 더 부드럽게 분산 (여러 곳에 골고루)

def attention_with_temperature(Q, K, V, temperature=1.0, mask=None):
    """
    Temperature가 추가된 Scaled Dot-Product Attention
    
    Args:
        temperature: Softmax 온도 (기본값=1.0)
                     높을수록 부드럽게, 낮을수록 날카롭게
    """
    d_k = Q.size(-1)
    # temperature를 나눠서 스케일 조정
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (math.sqrt(d_k) * temperature)
    if mask is not None:
        scores = scores.masked_fill(mask == 1, float('-inf'))
    weights = F.softmax(scores, dim=-1)
    output = torch.matmul(weights, V)
    return output, weights


# Temperature 실험: 다른 온도에서 Attention 분포 비교
seq_len = 6
d = 16
torch.manual_seed(0)

Q_t = torch.randn(1, seq_len, d)
K_t = torch.randn(1, seq_len, d)
V_t = torch.randn(1, seq_len, d)

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(18, 3))

for idx, temp in enumerate(temperatures):
    _, wts = attention_with_temperature(Q_t, K_t, V_t, temperature=temp)
    # 첫 번째 Query의 Attention 분포
    first_row = wts[0, 0].detach().numpy()
    
    axes[idx].bar(range(seq_len), first_row, color='steelblue', edgecolor='black')
    axes[idx].set_title(f'temp={temp}', fontsize=12, fontweight='bold')
    axes[idx].set_ylim(0, 1)
    axes[idx].set_xlabel('Key 위치')
    if idx == 0:
        axes[idx].set_ylabel('Attention Weight')
    entropy = -np.sum(first_row * np.log(first_row + 1e-10))
    axes[idx].set_xlabel(f'엔트로피={entropy:.2f}')

plt.suptitle('Temperature에 따른 Attention 분포 변화\n(낮은 temp=날카롭게, 높은 temp=부드럽게)', 
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('temperature_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("💡 엔트로피가 클수록 Attention이 더 골고루 분산됨")
print("   → Temperature가 높을수록 더 균등하게 주목")

## 📋 정리

### ✅ 이번 실습에서 배운 것

1. **Attention 함수 직접 구현** - Q, K, V 연산 직접 코딩
2. **학습 가능한 W_Q, W_K, W_V** - nn.Linear로 투영 레이어 추가
3. **Cross-Attention** - 서로 다른 시퀀스 간 Attention (번역에서 핵심)
4. **패딩 마스크** - 실제 토큰과 패딩을 구분하여 처리
5. **Temperature** - Attention 날카로움 조절 기법

### ➡️ 다음 실습 (065)
- **Causal (Masked) Attention** - 미래를 보지 못하게 마스킹
- GPT 같은 자기회귀(autoregressive) 언어 모델에서 필수!